# Import Packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq

# Fast Fourier Transform (FFT)

## 🔎 Quick Refresher: Fast Fourier Transform (FFT)

**What it is.**  
The FFT is an efficient algorithm to compute the Discrete Fourier Transform (DFT), converting a time-domain signal into its frequency-domain representation (complex spectrum).

**Why we use it (in ME data).**  
- Find dominant vibration/sound frequencies of machines  
- Detect harmonics/imbalances and potential faults  
- Extract frequency features for ML pipelines

**Key terms to recall.**  
- Sampling rate `fs` (Hz): samples per second  
- Number of samples `N`: determines frequency bin spacing `Δf = fs / N`  
- Nyquist frequency: `fs / 2` (highest resolvable frequency without aliasing)

**Common pitfalls.**  
- Mismatched `fs` ↔ time vector length  
- Forgetting single-sided scaling (×2 for non-DC, non-Nyquist bins)  
- Reading artifacts as “real” peaks when no window is used

## **Practice 1: FFT for Sine Function**

### **1-1. Define freqeuncies ($f$) and angular frequencies ($w$) to generate a sine function**

### What this cell sets up
- We define five test frequencies `f1…f5` (Hz) for a **synthetic multi-sine** signal.
- We also compute the corresponding **angular frequencies** `w = 2πf` (rad/s).
- These values will be used to build a ground-truth signal whose spectrum we already know, so we can verify the FFT.

**💡 Tip:** Keep test tones well below the Nyquist frequency (`fs/2`) to avoid aliasing in later steps.

In [ ]:
# Frequencies (Hz)
f1 = 2
f2 = 3
f3 = 5
f4 = 21
f5 = 30

# Angular frequencies = 2pi*f (rad/s)
w1 = 2*np.pi*f1
w2 = 2*np.pi*f2
w3 = 2*np.pi*f3
w4 = 2*np.pi*f4
w5 = 2*np.pi*f5

In [ ]:
print("f1 =",f1, "/ f2 =",f2, "/ f3 =",f3, "/ f4 =",f4, "/ f5 =",f5 )
print("w1 =",w1, "/ w2 =",w2, "/ w3 =",w3, "/ w4 =",w4, "/ w5 =",w5 )

---

### **1-2. Generate a sine function (discrete)**
(leave your frequency definitions as is; use/keep either `dt=0.01` for `fs=100 Hz` or `dt=0.001` for `fs=1000 Hz`)

### What this cell does
- Builds a **uniformly sampled time vector** `t` and a **sum of sines** `x(t)`.
- Duration `T = t[-1] - t[0] + dt` and sampling interval `dt` define the sampling rate `fs = 1/dt`.
- This controlled signal lets us check if FFT peaks appear at `f1…f5` with the correct amplitudes.

**Checks:**
- `len(t)` should match `len(x)`.
- Frequencies must satisfy `max(f_i) < fs/2` (Nyquist) to prevent aliasing.

In [ ]:
# --- Time vector & synthetic signal (discrete) ---
# Choose one of the following two lines:
t = np.arange(0, 5, 0.01)   # fs = 100 Hz, duration = 5 s
# t = np.arange(0, 5, 0.001)  # fs = 1000 Hz, duration = 5 s

# Multi-sine (known frequencies and amplitudes)
x = (5*np.sin(w1*t)
     + 4*np.sin(w2*t)
     + 3*np.sin(w3*t)
     + 2*np.sin(w4*t)
     + 1*np.sin(w5*t))

t.shape, x.shape


---

### **1-3. Plot the time domain signal**

### Why plot time domain first?
- Sanity check: amplitude levels, period, and overall duration.
- If the time series looks clipped, flat, or too short, fix that **before** running the FFT.


In [ ]:
plt.figure(figsize=(10,3))
plt.plot(t, x, 'b-')
plt.xlabel('t (sec)',fontsize=20)
plt.ylabel('x(t)',fontsize=20)
# plt.xlim(1,2)
plt.grid()
plt.show()

.

.

---

### **1-4. Implement the FFT**

#### 1-4-1) Compute the sampling frequency `fs`
### Note on `fs`
Instead of integer tricks, compute `fs = 1/mean(Δt)`.  
This avoids rounding issues when `t` is created with floating-point steps.

In [ ]:
dt = np.mean(np.diff(t))          # average spacing (handles small float errors)
fs = 1.0 / dt
fs

#### 1-4-2) FFT implementation (using `scipy.fft`)

### What happens here
- `fft(x)` returns a complex, **two-sided** spectrum (negative & positive frequencies).
- We convert it to a **single-sided** (positive) spectrum for easier interpretation.
- **Scaling matters**: for amplitude-correct single-sided plots,
  - multiply by 2 for interior bins,
  - **do not** double DC and Nyquist.
- `phase_deg` holds the phase of each positive-frequency bin (useful for reconstruction/diagnostics).


In [ ]:
N = len(x)                       # number of samples
X = fft(x)                       # complex spectrum (two-sided)
freq = fftfreq(N, 1/fs)          # matching frequency bins (Hz), two-sided

X[:10]

In [ ]:
# Keep the positive frequencies (single-sided spectrum)
k_pos = N//2           # number of positive frequencies
f_pos = freq[:k_pos]
X_pos = X[:k_pos]

# Amplitude scaling for single-sided magnitude spectrum
amp = (2 / N) * np.abs(X_pos)    # double all non-DC/non-Nyquist bins
amp[0] = amp[0] / 2              # DC should not be doubled
amp[-1] = amp[-1] / 2            # Nyquist should not be doubled

### ⚡ Special cases in the single-sided spectrum

- **DC bin (0 Hz)**  
  The very first FFT bin. It represents the average (constant) value of the signal.  
  → It has no negative-frequency partner, so **do not double it**.

- **Nyquist bin (fs/2 Hz, only when N is even)**  
  The very last FFT bin at half the sampling rate.  
  → It also has no negative-frequency partner, so **do not double it**.

- **All other positive-frequency bins**  
  Each has a matching negative-frequency bin.  
  → To preserve total energy when showing only the positive side, **double their amplitudes**.

#### 1-4-3) Plot the FFT result

- Peaks should appear near the known tones (`f1…f5`) with the expected amplitude scaling.

In [ ]:
plt.figure(figsize=(12,3))
plt.plot(f_pos, amp, 'r-')
plt.xlim(0, fs/2)                  # only up to Nyquist
plt.xlabel('Frequency (Hz)', fontsize=12)
plt.ylabel('Amplitude (single-sided)', fontsize=12)
plt.grid()
plt.tight_layout()
plt.show()

---

.

.

.



## **Practice 2: FFT for Example Data**

### **2-1. Load data**


In [ ]:
Data = pd.read_csv('https://github.com/ljwg3000/UNT_MEEN-AI-Fall2026/blob/main/AI_tutorial/DA1/ExampleData?raw=true', sep=',', header=None)
Data

### **2-2. Check the time domain graph**

In [ ]:
plt.plot(Data.iloc[:,0], Data.iloc[:,1]) # Select one sensor signal
plt.xlabel('Time(s)')
plt.ylabel('Acceleration(g)')
plt.grid()
plt.show()

### **2-3. Implement the FFT**

#### 2-3-1. Declare time (`t`) and data (`x`)

**What this cell does**
- `t` = the time vector (from the first column).
- `x` = the chosen sensor signal (from the second column).
- This prepares the input arrays for FFT.

In [ ]:
t = Data.iloc[:,0].values # Select a time   array of data
x = Data.iloc[:,1].values # Select a signal array of data

#### **2-3-2. Calculate sampling frequency (`fs`)**

- The sampling frequency `fs` is needed to convert FFT bin indices into actual frequencies (Hz).
- We compute:
  - `dt = mean difference in t` (accounts for floating-point errors),
  - `fs = 1/dt` (samples per second).
- Correct `fs` ensures the frequency axis matches the physical signal.

In [ ]:
dt = np.mean(np.diff(t))
fs = 1.0 / dt
fs

#### **2-3-3. FFT implementation**

- Compute the FFT:
  * `X = fft(x)` → complex spectrum (two-sided).
  * `freq = fftfreq(N, 1/fs)` → frequency bins (Hz).
- Extract the single-sided spectrum:
  * Keep only `0 … Nyquist (fs/2)` frequencies.
- Amplitude scaling:
  * `(2/N) * |X|` doubles non-DC/non-Nyquist bins (to account for symmetric energy).
  * DC bin (0 Hz) and Nyquist bin (fs/2, if N is even) are not doubled.
- Result: amplitude spectrum that matches the physical signal’s true amplitudes.

In [ ]:
N = len(x)
X = fft(x)
freq = fftfreq(N, 1/fs)

# single-sided
k_pos = N//2
f_pos = freq[:k_pos]
X_pos = X[:k_pos]

# Amplitude scaling
amp = (2 / N) * np.abs(X_pos)
amp[0] = amp[0] / 2
amp[-1] = amp[-1] / 2

#### **2-3-4. Plot FFT result**

**How to read this plot**
- The x-axis shows frequency (Hz) up to the Nyquist frequency (`fs/2`).
- The y-axis shows amplitude of each frequency component.
- Peaks indicate dominant periodicities in the sensor data.
- Comparing time-domain and frequency-domain views helps identify hidden periodic behavior.

In [ ]:
plt.figure(figsize=(12,3))
plt.plot(f_pos, amp, 'r-')
plt.xlim(0, fs/2)
plt.xlabel('Frequency (Hz)', fontsize=12)
plt.ylabel('Amplitude', fontsize=12)
plt.grid()
plt.tight_layout()
plt.show()

#### **2-3-5. Find the max peak**
- To get the max peak index, try `np.argmax(amp)`.

In [ ]:
peak_idx  = np.argmax(amp)
peak_freq = f_pos[peak_idx]
peak_amp  = amp[peak_idx]

print("Dominant peak frequency (Hz):", peak_freq)
print("Peak amplitude:", peak_amp)

---
.

.

## Mini Quiz: Find the dominant peak of the **Current** signal

**Q. Find the largest peak frequency from the FFT of the _Current_ data (the **last sensor column**).**

* Index the **last column** of `Data` as the current signal.
* Compute `fs` from the time vector `t = Data.iloc[:,0].values`.
* Perform FFT (single-sided, amplitude-scaled as in the section above), **plot** the spectrum, and find:
  * the **peak frequency** (Hz) with the largest amplitude
  * the **peak amplitude**
* Get the max peak using `np.argmax(amp)`.  

In [ ]:
t =
x =    # Current signal (last column)

dt =
fs =

# FFT




# Plot spectrum
plt.figure(figsize=(12,3))
plt.plot(f_pos, amp, 'r-')
plt.xlim(0, fs/2)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Amplitude')
plt.title('FFT Spectrum of Current')
plt.grid()
plt.tight_layout()
plt.show()

# Find the max peak
peak_idx  =
peak_freq =
peak_amp  =

print("Dominant peak frequency (Hz):", peak_freq)
print("Peak amplitude:", peak_amp)

<details>
<summary>Click to see Answer</summary>

```python
t = Data.iloc[:, 0].values
x = Data.iloc[:, -1].values   # Current signal (last column)

dt = np.mean(np.diff(t))
fs = 1.0 / dt

# FFT (single-sided)
N = len(x)
X = fft(x)
freq = fftfreq(N, d=1/fs)

k_pos = N // 2
f_pos = freq[:k_pos]
X_pos = X[:k_pos]

amp = (2.0 / N) * np.abs(X_pos)
amp[0] = amp[0] / 2
amp[-1] = amp[-1] / 2  

# Plot spectrum
plt.figure(figsize=(12,3))
plt.plot(f_pos, amp, 'r-')
plt.xlim(0, fs/2)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Amplitude')
plt.title('FFT Spectrum of Current')
plt.grid()
plt.tight_layout()
plt.show()


# Find the peak
peak_idx = np.argmax(amp)
peak_freq = f_pos[peak_idx]
peak_amp = amp[peak_idx]

print("Dominant peak frequency (Hz):", peak_freq)
print("Peak amplitude:", peak_amp)

.

.

### 🔎 Using `find_peaks` for FFT spectra
- The function `scipy.signal.find_peaks()` automatically detects local maxima in a signal.  
- For FFT analysis:
  * Input: amplitude spectrum (`amp`)  
  * Output: indices of peaks and optional properties (`height`, `prominence`, etc.)  
- Example: `peaks, props = find_peaks(amp, height=0.5)`  
  → returns all peaks above amplitude = 0.5.  
- Once indices are obtained, map them back to frequency: `peak_freqs = f_pos[peaks]`.  

In [ ]:
from scipy.signal import find_peaks

# suppose we already have f_pos (Hz) and amp (amplitude spectrum)
peaks, props = find_peaks(amp, height=0.5)  # only peaks above 0.5

# peak frequencies & amplitudes
peak_freqs = f_pos[peaks]
peak_amps = amp[peaks]

print("Peak frequencies:", peak_freqs)
print("Peak amplitudes:", peak_amps)

# plot with peaks marked
plt.figure(figsize=(12,3))
plt.plot(f_pos, amp, 'r-')
plt.plot(peak_freqs, peak_amps, 'bo')   # mark peaks
plt.xlim(0, 600)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude")
plt.title("FFT spectrum with detected peaks")
plt.grid(True)
plt.show()

### ⚡ Harmonic frequencies in this signal
- The strongest peak appears at **60 Hz** (the fundamental frequency).  
- Additional peaks are observed at **3rd (≈180 Hz)** and **5th (≈300 Hz)** harmonics.  
- Notice that **even-order harmonics (2nd, 4th, …)** are absent.  

👉 This is expected for **three-phase current signals**, which exhibit symmetry such that **even harmonics cancel out**, leaving only **odd-order harmonics** (3rd, 5th, …).  